<a href="https://colab.research.google.com/github/G-725/cxr-rrg/blob/colab-backend/CXR_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =======================
# FIXED SINGLE-CELL SERVER
# =======================

# ---------- INSTALL ----------
!pip install -q flask flask-cors pyngrok torchxrayvision scikit-image transformers accelerate huggingface_hub pillow

# ---------- IMPORTS ----------
import torch
import io
import numpy as np
import skimage.transform
import torchxrayvision as xrv

from flask import Flask, request, jsonify
from flask_cors import CORS

from transformers import pipeline
from huggingface_hub import login
from PIL import Image
import os
from pyngrok import ngrok

# Replace "YOUR_NGROK_AUTH_TOKEN" with your actual ngrok authentication token.
# You can find your ngrok auth token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN", "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
# ---------- LOGIN ----------
HF_TOKEN = "xxxxxxxxxxxxxxxxxxxxxxxx"
login(token=HF_TOKEN)

# ---------- LOAD X-RAY PATHOLOGY MODEL ----------
print("⏳ Loading Chest X-ray pathology model...")
xray_model = xrv.models.DenseNet(weights="densenet121-res224-chex")
xray_model.eval()
print("✅ X-ray pathology model loaded")

# ---------- LOAD MEDGEMMA ----------
print("⏳ Loading MedGemma-4B-IT (2–3 minutes)...")
pipe = pipeline(
    "image-text-to-text",
    model="google/medgemma-4b-it",
    torch_dtype=torch.bfloat16,
    device=0,
    trust_remote_code=True,
    token=HF_TOKEN
)
print("✅ MedGemma loaded")

# ---------- PATHOLOGY DETECTION ----------
def detect_pathologies(pil_img):
    img = np.array(pil_img.convert("L"))
    img = img / 255.0
    img = skimage.transform.resize(img, (224, 224))
    img = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float()

    with torch.no_grad():
        preds = xray_model(img)[0]

    labels = xray_model.pathologies
    return [labels[i] for i, v in enumerate(preds) if v > 0.30]

# ---------- REPORT GENERATION ----------
def generate_report(image_bytes):
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    detected = detect_pathologies(image)

    if detected:
        abnormality_hint = ", ".join(detected)
        rule = "DO NOT state the X-ray is normal."
    else:
        abnormality_hint = "No high-confidence abnormality detected, subtle pathology may exist."
        rule = "Normal findings allowed only if justified."

    prompt = f"""
You are a senior radiologist.

Preliminary AI screening suggests:
{abnormality_hint}

IMPORTANT:
- {rule}
- Be clinically decisive
- Avoid vague language

Write a detailed chest X-ray report using EXACTLY this structure:

1. **FINDINGS**
- **Lungs**:
- **Heart**:
- **Pleura**:
- **Bones**:

2. **IMPRESSION**
Concise clinical summary.

3. **DIAGNOSIS**
Single most likely diagnosis.
"""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]
    }]

    output = pipe(messages, max_new_tokens=700)
    return output[0]["generated_text"][-1]["content"]

# ---------- FLASK APP ----------
app = Flask(__name__)
CORS(app)   # ✅ app exists now

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"}), 200

@app.route("/predict", methods=["POST"])
def predict():
    try:
        if len(request.files) == 0:
            return jsonify({
                "status": "error",
                "message": "No file received"
            }), 400

        # ✅ accept ANY key: image, file, upload, etc.
        uploaded_file = list(request.files.values())[0]
        image_bytes = uploaded_file.read()

        report = generate_report(image_bytes)

        return jsonify({
            "status": "success",
            "report": report
        }), 200

    except Exception as e:
        return jsonify({
            "status": "error",
            "message": str(e)
        }), 500

# ---------- START SERVER + NGROK ----------
public_url = ngrok.connect(5000, bind_tls=True)
print("🚀 PUBLIC HTTPS URL:", public_url)

app.run(host="0.0.0.0", port=5000)
